In [ ]:
"""
ERL_posthoc_and_ecdf.py

1) Post-hoc within-field contrasts in normalized RSF risk among fumigation rates
   using balanced subsampling within flight, flight fixed effects, and flight-clustered SEs,
   plus R replicate draws for stability.

2) ECDF plots by field (overlaying all available treatments in that field).

Reads shapefiles from:
  F:\2023_El_Rio_Lobo\RSF_ERL\field_1\rsf_risk_exports\
  F:\2023_El_Rio_Lobo\RSF_ERL\field_2\rsf_risk_exports\
  F:\2023_El_Rio_Lobo\RSF_ERL\field_3\rsf_risk_exports\

Treatment field name can be either: tmt_lbs or trt_lbs
Treatment values: "250", "300", "350" lbs/acre
Mapped to kg/ha: 250->280, 300->336, 350->392
"""

from __future__ import annotations

from pathlib import Path
from typing import Optional, List, Tuple
import re
import numpy as np
import pandas as pd
import geopandas as gpd
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt


# ----------------------------
# CONFIG
# ----------------------------

from config import *

DATE_COL = "Date"
FIELDID_COL = "field_id"

# Shapefile exports can rename the actual field name; alias is not used in Python.
RISK_NORM_CANDIDATES = [
    "risk_norm",
    "risk_norm_",
    "risk_norm_1",
    "risk_norm__",
    "risknorm",
]

# Treatment lbs/acre field name varies across files
TMT_LBS_CANDIDATES = [
    "tmt_lbs",
    "trt_lbs",
    "TMT_LBS",
    "TRT_LBS",
    "tmtlbs",
    "trtlbs",
]

# Date window (optional). Set to None to use all.
START_DATE = pd.Timestamp("2023-05-15")
END_DATE = pd.Timestamp("2023-06-10")

# Mapping lbs/acre -> kg/ha
LBS_TO_KGHA = {250: 280, 300: 336, 350: 392}

# Balanced subsampling replicates
R = 50
BASE_SEED = 42

# ECDF output settings
ECDF_DPI = 300
ECDF_SAVE_PDF = True

# ----------------------------
# Helpers
# ----------------------------
def pick_first_existing_col(columns: List[str], candidates: List[str]) -> Optional[str]:
    col_map = {c.lower(): c for c in columns}
    for cand in candidates:
        if cand.lower() in col_map:
            return col_map[cand.lower()]
    return None


def parse_date(series: pd.Series) -> pd.Series:
    return pd.to_datetime(series, errors="coerce")


def normalize_field_id_to_int(x: object) -> Optional[int]:
    """Coerce 1, 1.0, '1', '1.0' -> 1."""
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return None
    s = str(x).strip()
    if s == "":
        return None
    try:
        return int(float(s))
    except Exception:
        return None


def normalize_lbs_to_int(x: object) -> Optional[int]:
    """Coerce 250, '250', '250.0', ' 350 ' -> int."""
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return None
    s = str(x).strip()
    if s == "":
        return None
    s = re.sub(r"[^0-9.]+", "", s)
    if s == "":
        return None
    try:
        return int(round(float(s)))
    except Exception:
        return None


def balanced_subsample_within_flight(df: pd.DataFrame, treat_col: str, flight_col: str, seed: int) -> pd.DataFrame:
    """
    Balance two treatment groups WITHIN each flight:
    for each flight, sample min(nA, nB) from each treatment.
    """
    rng = np.random.default_rng(seed)
    treatments = [t for t in df[treat_col].dropna().unique()]
    if len(treatments) != 2:
        raise ValueError(f"Expected exactly 2 treatments; got {treatments}")

    t1, t2 = treatments[0], treatments[1]
    parts = []

    for flight, g in df.groupby(flight_col):
        g1 = g[g[treat_col] == t1]
        g2 = g[g[treat_col] == t2]
        m = min(len(g1), len(g2))
        if m == 0:
            continue

        idx1 = rng.choice(g1.index.to_numpy(), size=m, replace=False)
        idx2 = rng.choice(g2.index.to_numpy(), size=m, replace=False)
        parts.append(df.loc[np.concatenate([idx1, idx2])])

    if not parts:
        return df.iloc[0:0].copy()

    return pd.concat(parts, ignore_index=True)


def fit_clustered_ols(df: pd.DataFrame, y_col: str, treat_col: str, flight_col: str) -> dict:
    """
    OLS with flight fixed effects and flight-clustered SE:
      y ~ C(treat) + C(flight)
    Assumes df contains exactly 2 treatment levels.
    """
    d = df.copy()
    d["_flight"] = pd.to_datetime(d[flight_col]).dt.strftime("%Y-%m-%d")

    # Force baseline to be the lower numeric rate
    levels = sorted(d[treat_col].unique())
    d["_treat"] = pd.Categorical(d[treat_col], categories=levels, ordered=True)

    model = smf.ols(f"{y_col} ~ C(_treat) + C(_flight)", data=d).fit(
        cov_type="cluster",
        cov_kwds={"groups": d["_flight"]},
    )

    treat_terms = [p for p in model.params.index if p.startswith("C(_treat)[T.")]
    if len(treat_terms) != 1:
        raise RuntimeError(f"Unexpected treatment terms: {treat_terms}")

    term = treat_terms[0]
    effect = float(model.params[term])
    se = float(model.bse[term])
    ci_low, ci_high = effect - 1.96 * se, effect + 1.96 * se

    ref = levels[0]
    other = levels[1]

    return {
        "contrast": f"{other} vs. {ref}",
        "effect": effect,
        "ci_low": ci_low,
        "ci_high": ci_high,
        "n": int(model.nobs),
        "n_flights": int(d["_flight"].nunique()),
    }


# ----------------------------
# IO + Preparation
# ----------------------------
def read_shapefiles_to_df() -> pd.DataFrame:
    shp_files: List[Path] = []
    for d in FIELD_DIRS:
        if d.exists():
            shp_files.extend(sorted(d.glob("*.shp")))
        else:
            print(f"[WARN] Missing folder: {d}")

    if not shp_files:
        raise FileNotFoundError("No shapefiles found in FIELD_DIRS.")

    parts = []
    print(f"Found {len(shp_files)} shapefiles.\n")

    for shp in shp_files:
        try:
            gdf = gpd.read_file(shp)

            missing = [c for c in (DATE_COL, FIELDID_COL) if c not in gdf.columns]
            if missing:
                raise KeyError(f"Missing required fields {missing}")

            risk_col = pick_first_existing_col(list(gdf.columns), RISK_NORM_CANDIDATES)
            if risk_col is None:
                raise KeyError(f"No risk_norm-like field found. Columns={list(gdf.columns)}")

            lbs_col = pick_first_existing_col(list(gdf.columns), TMT_LBS_CANDIDATES)
            if lbs_col is None:
                raise KeyError(f"No tmt_lbs/trt_lbs field found. Columns={list(gdf.columns)}")

            df = pd.DataFrame({
                "field_id_raw": gdf[FIELDID_COL],
                "Date": parse_date(gdf[DATE_COL]),
                "risk_norm": pd.to_numeric(gdf[risk_col], errors="coerce"),
                "tmt_lbs_raw": gdf[lbs_col],
            })

            parts.append(df)
            print(f"[OK] {shp.name}  (risk={risk_col}, lbs={lbs_col})")

        except Exception as e:
            print(f"[WARN] Skipping {shp.name}: {e}")

    if not parts:
        raise RuntimeError("No shapefiles were read successfully.")

    return pd.concat(parts, ignore_index=True)


def prepare_analysis_df(df_all: pd.DataFrame) -> pd.DataFrame:
    d = df_all.copy()

    # Date window
    if START_DATE is not None:
        d = d[d["Date"] >= START_DATE]
    if END_DATE is not None:
        d = d[d["Date"] <= END_DATE]

    d = d.dropna(subset=["Date", "risk_norm", "field_id_raw", "tmt_lbs_raw"]).copy()

    # Normalize IDs
    d["field_id"] = d["field_id_raw"].map(normalize_field_id_to_int)
    d["tmt_lbs"] = d["tmt_lbs_raw"].map(normalize_lbs_to_int)

    d = d.dropna(subset=["field_id", "tmt_lbs"]).copy()
    d["field_id"] = d["field_id"].astype(int)
    d["tmt_lbs"] = d["tmt_lbs"].astype(int)

    # Map to kg/ha
    d["tmt_kgha"] = d["tmt_lbs"].map(LBS_TO_KGHA)
    d = d.dropna(subset=["tmt_kgha"]).copy()
    d["tmt_kgha"] = d["tmt_kgha"].astype(int)

    # Keep only expected lbs values (safety)
    d = d[d["tmt_lbs"].isin(LBS_TO_KGHA.keys())].copy()

    return d


# ----------------------------
# Post-hoc contrasts
# ----------------------------
def build_valid_contrasts(d: pd.DataFrame) -> List[Tuple[int, int, int]]:
    """
    Build within-field pairwise contrasts from treatments that exist in that field.
    Returns (field_id, higher_rate, lower_rate) in kg/ha.
    """
    contrasts: List[Tuple[int, int, int]] = []
    for fid, g in d.groupby("field_id"):
        rates = sorted(g["tmt_kgha"].dropna().unique().tolist())
        for i in range(len(rates)):
            for j in range(i):
                contrasts.append((int(fid), int(rates[i]), int(rates[j])))
    return contrasts


def run_contrast_with_replicates(d: pd.DataFrame, field_id: int, a: int, b: int) -> dict:
    """
    One within-field contrast (a vs b) with:
      - primary single balanced draw + clustered CI
      - replicate distribution across R draws
    Uses only flights where BOTH treatments exist.
    """
    sub = d[(d["field_id"] == field_id) & (d["tmt_kgha"].isin([a, b]))].copy()
    if sub.empty:
        return _empty_row(field_id, a, b)

    # Strict: keep only flight dates where both treatments exist
    counts = sub.groupby(["Date", "tmt_kgha"]).size().unstack(fill_value=0)
    if a not in counts.columns or b not in counts.columns:
        return _empty_row(field_id, a, b)

    ok_flights = counts[(counts[a] > 0) & (counts[b] > 0)].index
    sub = sub[sub["Date"].isin(ok_flights)].copy()
    if sub.empty:
        return _empty_row(field_id, a, b)

    # Primary
    sub_primary = balanced_subsample_within_flight(sub, "tmt_kgha", "Date", seed=BASE_SEED)
    primary = fit_clustered_ols(sub_primary, y_col="risk_norm", treat_col="tmt_kgha", flight_col="Date")

    # Replicates
    effects = []
    for r in range(R):
        sub_r = balanced_subsample_within_flight(sub, "tmt_kgha", "Date", seed=BASE_SEED + r + 1)
        if sub_r.empty:
            continue
        out_r = fit_clustered_ols(sub_r, y_col="risk_norm", treat_col="tmt_kgha", flight_col="Date")
        effects.append(out_r["effect"])

    effects = np.array(effects, dtype=float)
    if effects.size == 0:
        mean_eff = np.nan
        lo = np.nan
        hi = np.nan
        used_r = 0
    else:
        mean_eff = float(np.mean(effects))
        lo = float(np.quantile(effects, 0.025))
        hi = float(np.quantile(effects, 0.975))
        used_r = int(effects.size)

    return {
        "Field": field_id,
        "Treatment (kg/ha)": primary["contrast"],
        "Effect (primary)": primary["effect"],
        "95% CI (primary)": f"{primary['ci_low']:.3f}, {primary['ci_high']:.3f}",
        "n (primary)": primary["n"],
        "Flights (primary)": primary["n_flights"],
        "Effect (mean over R)": mean_eff,
        "Effect 2.5%": lo,
        "Effect 97.5%": hi,
        "R": used_r,
    }


def _empty_row(field_id: int, a: int, b: int) -> dict:
    return {
        "Field": field_id,
        "Treatment (kg/ha)": f"{a} vs. {b}",
        "Effect (primary)": np.nan,
        "95% CI (primary)": "",
        "n (primary)": 0,
        "Flights (primary)": 0,
        "Effect (mean over R)": np.nan,
        "Effect 2.5%": np.nan,
        "Effect 97.5%": np.nan,
        "R": 0,
    }


# ----------------------------
# ECDF plots
# ----------------------------
def ecdf_xy(values: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    x = np.sort(values)
    n = x.size
    y = np.arange(1, n + 1) / n
    return x, y


def plot_ecdf_by_field(
    d: pd.DataFrame,
    out_dir: Path,
    risk_col: str = "risk_norm",
    field_col: str = "field_id",
    treat_col: str = "tmt_kgha",
    dpi: int = 300,
    save_pdf: bool = True,
):
    out_dir.mkdir(parents=True, exist_ok=True)

    dd = d.dropna(subset=[field_col, treat_col, risk_col]).copy()
    dd[risk_col] = pd.to_numeric(dd[risk_col], errors="coerce")
    dd = dd.dropna(subset=[risk_col]).copy()

    dd[treat_col] = pd.to_numeric(dd[treat_col], errors="coerce")
    dd = dd.dropna(subset=[treat_col]).copy()
    dd[treat_col] = dd[treat_col].astype(int)

    for fid, g in dd.groupby(field_col):
        rates = sorted(g[treat_col].unique().tolist())
        if not rates:
            continue

        fig, ax = plt.subplots(figsize=(6.5, 4.5))

        for rate in rates:
            vals = g.loc[g[treat_col] == rate, risk_col].to_numpy(dtype=float)
            vals = vals[np.isfinite(vals)]
            if vals.size == 0:
                continue
            x, y = ecdf_xy(vals)
            ax.plot(x, y, linewidth=2, label=f"{rate} kg/ha")

        ax.set_xlabel("Normalized RSF risk (0–1)")
        ax.set_ylabel("ECDF (proportion ≤ x)")
        ax.set_xlim(0, 1)
        ax.set_ylim(0, 1)
        ax.grid(True, alpha=0.3)
        ax.legend(title="Fumigation rate", loc="lower right")

        # Title includes date window
        if START_DATE is not None and END_DATE is not None:
            ax.set_title(f"Field {fid} ECDF of normalized RSF risk\n({START_DATE:%Y-%m-%d} to {END_DATE:%Y-%m-%d})")
        else:
            ax.set_title(f"Field {fid} ECDF of normalized RSF risk")

        fig.tight_layout()

        png_path = out_dir / f"ERL_Field_{fid}_ECDF.png"
        fig.savefig(png_path, dpi=dpi)
        if save_pdf:
            pdf_path = out_dir / f"ERL_Field_{fid}_ECDF.pdf"
            fig.savefig(pdf_path)  # vector output

        plt.close(fig)
        print(f"[OK] Wrote ECDF for Field {fid}: {png_path}")


# ----------------------------
# Main
# ----------------------------
def main() -> None:
    df_all = read_shapefiles_to_df()
    d = prepare_analysis_df(df_all)

    # Post-hoc contrasts
    contrasts = build_valid_contrasts(d)
    if not contrasts:
        raise RuntimeError("No valid contrasts found. Check treatment mapping / data.")

    print(f"\nValid contrasts discovered: {len(contrasts)}")
    for fid, a, b in contrasts:
        print(f"  Field {fid}: {a} vs {b} (kg/ha)")

    rows = []
    for fid, a, b in contrasts:
        row = run_contrast_with_replicates(d, fid, a, b)
        rows.append(row)
        print(f"[OK] Field {fid} {row['Treatment (kg/ha)']}  n={row['n (primary)']:,}  flights={row['Flights (primary)']}  R={row['R']}")

    out = pd.DataFrame(rows)

    # Format numeric columns to 3 decimals for table-like output
    def fmt3(x):
        return "" if pd.isna(x) else f"{x:.3f}"

    out["Effect (primary)"] = out["Effect (primary)"].map(fmt3)
    out["Effect (mean over R)"] = out["Effect (mean over R)"].map(fmt3)
    out["Effect 2.5%"] = out["Effect 2.5%"].map(fmt3)
    out["Effect 97.5%"] = out["Effect 97.5%"].map(fmt3)

    out.to_csv(OUT_TABLE_CSV, index=False)
    print(f"\nWrote: {OUT_TABLE_CSV}\n")
    print(out.to_string(index=False))

    # ECDF plots (one per field)
    print("\nGenerating ECDF plots...")
    plot_ecdf_by_field(d, out_dir=ECDF_DIR, dpi=ECDF_DPI, save_pdf=ECDF_SAVE_PDF)
    print(f"\nECDF outputs in: {ECDF_DIR}\n")


if __name__ == "__main__":
    main()


Found 18 shapefiles.

[OK] ERL_Field_1_20230421_HexMetrics_WithRisk_TRT.shp  (risk=risk_norm_, lbs=trt_lbs)
[OK] ERL_Field_1_20230515_HexMetrics_WithRisk_TRT.shp  (risk=risk_norm_, lbs=trt_lbs)
[OK] ERL_Field_1_20230522_HexMetrics_WithRisk_TRT.shp  (risk=risk_norm_, lbs=trt_lbs)
[OK] ERL_Field_1_20230601_HexMetrics_WithRisk_TRT.shp  (risk=risk_norm_, lbs=trt_lbs)
[OK] ERL_Field_1_20230610_HexMetrics_WithRisk_TRT.shp  (risk=risk_norm_, lbs=trt_lbs)
[OK] ERL_Field_1_20230620_HexMetrics_WithRisk_TRT.shp  (risk=risk_norm, lbs=trt_lbs)
[OK] ERL_Field2_20230421_HexMetrics_WithRisk_TRT.shp  (risk=risk_norm_, lbs=tmt_lbs)
[OK] ERL_Field2_20230515_HexMetrics_WithRisk_TRT.shp  (risk=risk_norm_, lbs=tmt_lbs)
[OK] ERL_Field2_20230522_HexMetrics_WithRisk_TRT.shp  (risk=risk_norm_, lbs=tmt_lbs)
[OK] ERL_Field2_20230601_HexMetrics_WithRisk_TRT.shp  (risk=risk_norm_, lbs=tmt_lbs)
[OK] ERL_Field2_20230610_HexMetrics_WithRisk_TRT.shp  (risk=risk_norm_, lbs=tmt_lbs)
[OK] ERL_Field2_20230620_HexMetrics_Wi